In [ ]:
#estimate background rms
from astropy.io import fits
import numpy as np

# Open FITS safely and read data/header
with fits.open('../data/radio images/lotss1_0.03100_23.395000_r.fits') as hd:
    data = hd[0].data
    hdr  = hd[0].header

# Squeeze extra axes if present
if data is not None and data.ndim > 2:
    data = data.squeeze()

# Build mask of finite pixels (exclude NaN/Inf)
finite = np.isfinite(data)
finite_vals = data[finite]
if finite_vals.size == 0:
    raise ValueError('No finite pixels found in image. Check FITS data.')

# Robust background using MAD; exclude the brightest 0.5% to avoid source/core bias
thr = np.percentile(finite_vals, 99.5)
bgpix = finite_vals[finite_vals < thr]
if bgpix.size < 100:  # fallback if filtering leaves too few pixels
    bgpix = finite_vals

med = np.median(bgpix)
mad = np.median(np.abs(bgpix - med))
rms = 1.4826 * mad

peak = finite_vals.max()

print('Estimated RMS = {:.3e} (same units as FITS)'.format(rms))
print('Peak value = {:.3e}'.format(peak))
print('Background sample size = {} / {} finite pixels'.format(bgpix.size, finite_vals.size))
print('99.5th percentile threshold (exclusion) = {:.3e}'.format(thr))

# Recommended contour sets:
levels_sigma = np.array([3,5,8,12,20]) * rms
levels_geo   = 3 * rms * (2.0 ** np.arange(0,6))   # 3,6,12,24...
print('Contour levels (sigma-list):', levels_sigma)
print('Contour levels (geometric):', levels_geo)


Estimated RMS = 9.360e-05 (same units as FITS)
Peak value = 2.146e+00
Background sample size = 62429845 / 62743563 finite pixels
99.5th percentile threshold (exclusion) = 4.239e-04
Contour levels (sigma-list): [0.00028081 0.00046801 0.00074881 0.00112322 0.00187203]
Contour levels (geometric): [0.00028081 0.00056161 0.00112322 0.00224644 0.00449288 0.00898576]
